In [1]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import pandas as pd
import numpy as np

In [2]:
EXPERIMENT_NAME = "demo_mlflow_edu"
MODEL_NAME = "iris_classifier_edu"

mlflow.set_tracking_uri("sqlite:///mlflow_edu.db")

experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
mlflow.set_experiment(EXPERIMENT_NAME)

2026/03/10 12:25:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/10 12:25:36 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='file:C:/Users/ana/Documents/Cursos/warmUpFIAP/1_MLFlow_Gerenciamento_/mlruns/1', creation_time=1773156337639, experiment_id='1', last_update_time=1773156337639, lifecycle_stage='active', name='demo_mlflow_edu', tags={}, workspace='default'>

In [3]:
iris = load_iris()

X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)

In [4]:
# Nested runs
# Parent run
parent_run = mlflow.start_run(run_name="parent_run_model_comparison")

mlflow.log_param("experiment_type", "model_comparison")
mlflow.log_param("models_tested", 2)
mlflow.log_param("dataset_name", "iris")

best_model = None
best_accuracy = 0
best_run_id = None

In [5]:
parent_run.info.run_id

'7a08d0c910e74011a26d057e2e3aff79'

In [6]:
rf_params = [
    {"n_estimators": 50, "max_depth": 2},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": 7},
]

for params in rf_params:
    with mlflow.start_run(run_name=f"rf_nest_{params["n_estimators"]}_trees", nested=True):

        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        mlflow.log_params(params)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("n_features", X_train.shape[1])

        mlflow.sklearn.log_model(model, "random_forest_model")

        print(f"    RF with {params['n_estimators']} trees: {accuracy:.2f}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model
            best_run_id = mlflow.active_run().info.run_id


2026/03/10 12:25:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 12:25:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


    RF with 50 trees: 1.00


2026/03/10 12:25:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 12:25:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


    RF with 100 trees: 1.00


2026/03/10 12:25:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 12:25:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


    RF with 200 trees: 1.00


In [7]:
print(f"\n  Testing logistic regression model")

lr_params = [
    {"C": 0.1, "max_iter": 1000},
    {"C": 1, "max_iter": 1000},
    {"C": 10, "max_iter": 1000}
]

for params in lr_params:
    with mlflow.start_run(run_name=f"lr_nest_{params['C']}_C", nested=True):
        model = LogisticRegression(**params, random_state=42)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        mlflow.log_params(params)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("n_features", X_train.shape[1])

        mlflow.sklearn.log_model(model, "logistic_regression")

        print(f"    LR with C={params['C']}: {accuracy:.2f}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model
            best_run_id = mlflow.active_run().info.run_id

2026/03/10 12:26:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 12:26:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Testing logistic regression model


2026/03/10 12:26:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 12:26:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


    LR with C=0.1: 1.00


2026/03/10 12:26:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 12:26:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


    LR with C=1: 1.00
    LR with C=10: 1.00


In [8]:
# Closing parent run
mlflow.log_metric("best_accuracy", best_accuracy)
mlflow.set_tag("best_child_run_id", best_run_id)
mlflow.set_tag("best_model_type", type(best_model).__name__)

print(f"\n  Best accuracy: {best_accuracy:.2f}")
print(f"\n  Best model id: {best_run_id}")

mlflow.end_run()

print(f"Nested runs completed.")




  Best accuracy: 1.00

  Best model id: fddbba8ac94d4ee8a0c23996f1c6bf11
Nested runs completed.


In [9]:
# Autologging
mlflow.sklearn.autolog()

In [10]:
with mlflow.start_run(run_name="autolog_gridsearch"):
    param_grid = {
        "n_estimators": [50, 100, 200],
        "max_depth": [3, 5, 7],
        "min_samples_split": [2, 5],
    }

    rf = RandomForestClassifier(random_state=42)
    grid_search = GridSearchCV(rf, param_grid, cv=3, scoring="accuracy", n_jobs=1, verbose=1)
    grid_search.fit(X_train, y_train)

    predictions = grid_search.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average="weighted")
    recall = recall_score(y_test, predictions, average="weighted")
    f1 = f1_score(y_test, predictions, average="weighted")

    grid_run_id = mlflow.active_run().info.run_id

    print(f"\n  Results for grid search:")
    print(f"        Best configuration: {grid_search.best_params_}")
    print(f"        Best score CV: {grid_search.best_score_:.4f}")
    print(f"        Accuracy: {accuracy:.2f}")
    print(f"        Precision: {precision:.2f}")
    print(f"        Recall: {recall:.2f}")
    print(f"        F1 Score: {f1:.2f}")

    best_grid_model = grid_search.best_estimator_

Fitting 3 folds for each of 18 candidates, totalling 54 fits


2026/03/10 12:26:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/10 12:26:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/10 12:26:40 INFO mlflow.sklearn.utils: Logging the 5 best runs, 13 runs will be omitted.



  Results for grid search:
        Best configuration: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
        Best score CV: 0.9583
        Accuracy: 1.00
        Precision: 1.00
        Recall: 1.00
        F1 Score: 1.00


In [11]:
mlflow.sklearn.autolog(disable=True)

In [12]:
# Model registry and life cycle
client = MlflowClient()

model_uri = f"runs:/{grid_run_id}/model"

model_version = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(model_version.version)

Successfully registered model 'iris_classifier_edu'.
2026/03/10 12:26:40 WARNING mlflow.tracking._model_registry.fluent: Run with id dd91fa9117584bedbba075df88640e6a has no artifacts at artifact path 'model', registering model based on models:/m-f4af4452cbe94bbe97290aee5f09dc65 instead


1


Created version '1' of model 'iris_classifier_edu'.


In [13]:
client.update_model_version(
    name=MODEL_NAME,
    version=model_version.version,
    description="Classification model for iris dataset optimized with GridSearchCV"
)

tags_to_add = {
    "algorithm": "RandomForest",
    "dataset": "iris",
    "framework": "scikit-learn",
    "optimizer": "GridSearchCV",
    "author": "MLFlow Demo"
}

for key, value in tags_to_add.items():
    client.set_model_version_tag(
        name=MODEL_NAME,
        version=model_version.version,
        key=key,
        value=value
    )
    print(f"    Tag {key}={value} added to model version {model_version.version}")

    Tag algorithm=RandomForest added to model version 1
    Tag dataset=iris added to model version 1
    Tag framework=scikit-learn added to model version 1
    Tag optimizer=GridSearchCV added to model version 1
    Tag author=MLFlow Demo added to model version 1


In [14]:
# Proceeding to staging
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=model_version.version,
    stage="Staging",
    archive_existing_versions=False
)

C:\Users\ana\AppData\Local\Temp\ipykernel_3132\3382611164.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773156400782, current_stage='Staging', deployment_job_state=None, description='Classification model for iris dataset optimized with GridSearchCV', last_updated_timestamp=1773156400971, metrics=None, model_id=None, name='iris_classifier_edu', params=None, run_id='dd91fa9117584bedbba075df88640e6a', run_link=None, source='models:/m-f4af4452cbe94bbe97290aee5f09dc65', status='READY', status_message=None, tags={'algorithm': 'RandomForest',
 'author': 'MLFlow Demo',
 'dataset': 'iris',
 'framework': 'scikit-learn',
 'optimizer': 'GridSearchCV'}, user_id=None, version=1, workspace='default'>

In [15]:
# Validating model in staging
staging_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{MODEL_NAME}/Staging"
)

test_predictions = staging_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

validation_checks = {
    "performance_validation": test_accuracy > 0.9,
    "latency_validation": True,
    "business_impact_validation": True,
    "team_approval": True
}

all_passed = all(validation_checks.values())
print(f"\n{"Model passed" if all_passed else "Model failed"} validation checks: {validation_checks}")


Model passed validation checks: {'performance_validation': True, 'latency_validation': True, 'business_impact_validation': True, 'team_approval': True}


In [16]:
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=model_version.version,
    stage="Production",
    archive_existing_versions=True
)

C:\Users\ana\AppData\Local\Temp\ipykernel_3132\515635807.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773156400782, current_stage='Production', deployment_job_state=None, description='Classification model for iris dataset optimized with GridSearchCV', last_updated_timestamp=1773156401099, metrics=None, model_id=None, name='iris_classifier_edu', params=None, run_id='dd91fa9117584bedbba075df88640e6a', run_link=None, source='models:/m-f4af4452cbe94bbe97290aee5f09dc65', status='READY', status_message=None, tags={'algorithm': 'RandomForest',
 'author': 'MLFlow Demo',
 'dataset': 'iris',
 'framework': 'scikit-learn',
 'optimizer': 'GridSearchCV'}, user_id=None, version=1, workspace='default'>

In [17]:
# Usando modelo em produção
production_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{MODEL_NAME}/Production"
)

print("Model loaded successfully.")
print(f"    Type:       {type(production_model).__name__}")
print(f"    Parameters: {production_model.get_params()}")

Model loaded successfully.
    Type:       GridSearchCV
    Parameters: {'cv': 3, 'error_score': nan, 'estimator__bootstrap': True, 'estimator__ccp_alpha': 0.0, 'estimator__class_weight': None, 'estimator__criterion': 'gini', 'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__max_leaf_nodes': None, 'estimator__max_samples': None, 'estimator__min_impurity_decrease': 0.0, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'estimator__min_weight_fraction_leaf': 0.0, 'estimator__monotonic_cst': None, 'estimator__n_estimators': 100, 'estimator__n_jobs': None, 'estimator__oob_score': False, 'estimator__random_state': 42, 'estimator__verbose': 0, 'estimator__warm_start': False, 'estimator': RandomForestClassifier(random_state=42), 'n_jobs': 1, 'param_grid': {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 7], 'min_samples_split': [2, 5]}, 'pre_dispatch': '2*n_jobs', 'refit': True, 'return_train_score': False, 'scoring': 'accuracy', 'verbose': 1}

In [18]:
new_samples = np.array([
    [6.4, 3.2, 4.5, 1.5],
    [5.8, 2.7, 5.1, 1.9],
    [5.1, 3.5, 1.4, 0.2]
])

predictions = production_model.predict(new_samples)
probabilities = production_model.predict_proba(new_samples)

for i, (sample, prediction, probability) in enumerate(zip(new_samples, predictions, probabilities)):
    print(f"Sample {i+1}: {sample}")
    print(f"    Prediction: {prediction}")
    print(f"    Probabilities: {probability}")

Sample 1: [6.4 3.2 4.5 1.5]
    Prediction: 1
    Probabilities: [0.         0.99714286 0.00285714]
Sample 2: [5.8 2.7 5.1 1.9]
    Prediction: 2
    Probabilities: [0.         0.04307692 0.95692308]
Sample 3: [5.1 3.5 1.4 0.2]
    Prediction: 0
    Probabilities: [1. 0. 0.]


In [19]:
!mlflow ui --backend-store-uri sqlite:///mlflow_edu.db

^C
